# 02 · Team climate: within-team vs. between-team effects

**The question:** do soldiers report better well-being when their *team's* leadership climate is better, over and above their *own* view of their leader?

This is a question Culture Amp customers ask in other words: *"Is engagement about the manager, or about how each person experiences the manager?"* The answer changes what you recommend. A team-level effect points to the manager and the team's shared climate. An individual-level effect points to each person's own experience, which a team-wide intervention may not reach.

**By the end you'll be able to:**
1. Split a predictor into its within-team and between-team parts (centering)
2. Explain why one ordinary regression slope answers neither question
3. Estimate and interpret a contextual effect
4. Test whether a relationship differs across teams (random slopes)
5. Test *why* it differs (a cross-level interaction), and say how confident to be

In [ ]:
# colab: runtime > change runtime type > R. the first run installs packages (about a minute)
for (pkg in c("multilevel", "ggplot2")) {
  if (!requireNamespace(pkg, quietly = TRUE)) install.packages(pkg)
}
suppressPackageStartupMessages({
  library(multilevel)  # also loads nlme, which fits the models
  library(ggplot2)
})
options(digits = 3, repr.plot.width = 9, repr.plot.height = 4.5)
ctl <- lmeControl(opt = "optim")  # a sturdier optimizer for models with random slopes

# checks your answers in the "your turn" cells
check <- function(yours, target, label = "answer", tol = 0.005) {
  if (is.null(yours)) return(message("[todo] ", label, ": not filled in yet"))
  keep <- !is.na(target)
  gap <- abs(as.numeric(yours)[keep] - as.numeric(target)[keep])
  if (length(yours) == length(target) && !anyNA(gap) && max(gap) <= tol) {
    message("[ok] ", label, ": ", paste(round(head(as.numeric(yours), 3), 3), collapse = ", "))
  } else {
    message("[not yet] ", label, ": yours = ", paste(round(head(as.numeric(yours), 3), 3), collapse = ", "),
            ". recheck the formula")
  }
}

data(bh1996)

## Part A · One predictor, two questions

Any individual rating can be split exactly into two pieces:

$$\text{LEAD}_{ij} = \underbrace{(\text{LEAD}_{ij} - \overline{\text{LEAD}}_j)}_{\text{within: me vs. my team}} + \underbrace{\overline{\text{LEAD}}_j}_{\text{between: my team vs. other teams}}$$

The within piece asks: *when a soldier rates leadership higher than their teammates do, is their well-being higher than their teammates'?*
The between piece asks: *do teams with better leadership climate have better average well-being?*

These are different questions, and nothing forces them to have the same answer.

In [ ]:
# your turn: rebuild the team mean and the within-team score for leadership
# hint: ave(x, group) returns, for every row, the mean of x in that row's group
teamLead <- NULL     # each soldier's team-average leadership rating
withinLead <- NULL   # soldier's rating minus their team's average

check(teamLead, bh1996$G.LEAD, "team mean (G.LEAD)")
check(withinLead, bh1996$W.LEAD, "within-team score (W.LEAD)")

In [ ]:
# by construction, the within piece averages zero and is uncorrelated with the team mean
round(c(meanOfWithin = mean(bh1996$W.LEAD), corWithinVsTeam = cor(bh1996$W.LEAD, bh1996$G.LEAD)), 4)

## Part B · Three models, one variable

| Model | Predictors | What the leadership coefficients mean |
|---|---|---|
| 1 | `LEAD` | a blend of the within and between relationships |
| 2 | `W.LEAD` + `G.LEAD` | within slope; between slope |
| 3 | `LEAD` + `G.LEAD` | within slope; **contextual effect** = between − within |

**Predict first:** will the between-team slope (do better-led teams have better well-being?) be **larger** or **smaller** than the within-team slope?

In [ ]:
olsRaw <- lm(WBEING ~ LEAD, data = bh1996)
rawOnly <- lme(WBEING ~ LEAD, random = ~ 1 | GRP, data = bh1996)
withinBetween <- lme(WBEING ~ W.LEAD + G.LEAD, random = ~ 1 | GRP, data = bh1996)
contextual <- lme(WBEING ~ LEAD + G.LEAD, random = ~ 1 | GRP, data = bh1996)

coefRow <- function(fit, term) {
  if (inherits(fit, "lm")) {
    tt <- coef(summary(fit)); return(unname(tt[term, c("Estimate", "Std. Error", "Pr(>|t|)")]))
  }
  tt <- summary(fit)$tTable; unname(tt[term, c("Value", "Std.Error", "p-value")])
}
results <- rbind(
  "ordinary regression: LEAD"    = coefRow(olsRaw, "LEAD"),
  "model 1: LEAD"                = coefRow(rawOnly, "LEAD"),
  "model 2: within (W.LEAD)"     = coefRow(withinBetween, "W.LEAD"),
  "model 2: between (G.LEAD)"    = coefRow(withinBetween, "G.LEAD"),
  "model 3: LEAD"                = coefRow(contextual, "LEAD"),
  "model 3: G.LEAD (contextual)" = coefRow(contextual, "G.LEAD")
)
colnames(results) <- c("estimate", "stdError", "p")
round(results, 4)

Read the table row by row:

- **Within ≈ 0.52.** A soldier who rates leadership one point higher than their teammates reports about half a point higher well-being than those teammates.
- **Between ≈ 0.41.** Teams whose average leadership rating is one point higher have about 0.41 points higher average well-being.
- **Model 3's `LEAD` equals model 2's within slope.** Holding the team mean fixed, `LEAD` can only vary within a team.
- **Model 3's `G.LEAD` is the contextual effect**, between minus within ≈ −0.10, p ≈ .09. Once you know how a soldier personally sees their leader, the team's shared climate adds **no evidence** of anything more.
- **The ordinary regression slope (≈ 0.50)** is a weighted blend of the two. It answers neither question, and nothing in its output tells you so.

What you'd tell a customer: the people who rate leadership well also report better well-being, and better-led teams do too. But the team-level link is no stronger than the individual one, so this data doesn't show a shared "climate effect" beyond each person's own experience of their leader.

The same name, `G.LEAD`, meant "between slope" in model 2 and "contextual effect" in model 3. **Centering decides what a coefficient means.** Always say which version you ran.

In [ ]:
# your turn: compute the contextual effect yourself from model 2 (between slope minus within slope)
myContextual <- NULL

check(myContextual, fixef(contextual)["G.LEAD"], "contextual effect", tol = 0.002)

## Part C · Does the leadership effect differ across teams?

So far every team shared one within-team slope. A **random slope** lets each team have its own:

$$\text{WBEING}_{ij} = \beta_{0j} + \beta_{1j}\,\text{W.LEAD}_{ij} + \dots + r_{ij}$$
$$\beta_{0j} = \gamma_{00} + \gamma_{01}\,\text{G.LEAD}_j + \dots + u_{0j}, \qquad \beta_{1j} = \gamma_{10} + u_{1j}$$

$\gamma_{10}$ is the average within-team slope. The variance of $u_{1j}$ says how much teams differ around it. Work hours, split the same way, go in as controls.

To test whether slopes really vary, compare the two models with a likelihood-ratio test. One caveat: a variance can't go below zero, so this test is **conservative**. Many analysts halve the p-value.

In [ ]:
fixedPart <- WBEING ~ W.HRS + W.LEAD + G.HRS + G.LEAD
interceptOnly <- lme(fixedPart, random = ~ 1 | GRP, data = bh1996)
randomSlope <- lme(fixedPart, random = ~ W.LEAD | GRP, data = bh1996, control = ctl)

anova(interceptOnly, randomSlope)
VarCorr(randomSlope)

The slope variance is clearly real. Both models have the same fixed effects, so comparing their REML fits is fine. When two models differ in **fixed** effects, refit both with `method = "ML"` before comparing them.

How much do teams differ? If team slopes are roughly normal, 95% of them fall within the average slope ± 1.96 × the slope's standard deviation.

In [ ]:
slopeSd <- as.numeric(VarCorr(randomSlope)["W.LEAD", "StdDev"])
avgSlope <- unname(fixef(randomSlope)["W.LEAD"])

# your turn: the low and high ends of the range that holds 95% of team slopes
slopeRange <- NULL   # c(low, high)

check(slopeRange, avgSlope + c(-1.96, 1.96) * slopeSd, "95% range of team leadership slopes")

In some teams, leadership ratings track well-being about twice as strongly as in others. Every team still has a positive slope.

**A centering warning.** Refit the same model with *raw* `LEAD` as the random slope and look at the intercept–slope correlation.

In [ ]:
rawSlope <- lme(WBEING ~ W.HRS + LEAD + G.HRS + G.LEAD, random = ~ LEAD | GRP, data = bh1996, control = ctl)
VarCorr(rawSlope)

A correlation near −1 is a warning sign. With raw `LEAD`, each team's "intercept" is its predicted well-being at a leadership rating of **zero**, which is off the 1–5 scale. Intercept and slope then become nearly the same piece of information, and the estimates get unstable. Centering puts the intercept somewhere meaningful (the team's average soldier), and the correlation drops to a sensible value.

## Part D · *Why* do slopes differ? A cross-level interaction

**The hypothesis:** leadership matters more for well-being in teams under heavier demands, here teams that work longer hours. To test it, let the team's average hours predict the size of the within-team leadership slope:

$$\beta_{1j} = \gamma_{10} + \gamma_{11}\,\text{G.HRS}_j + u_{1j}$$

In R, that's a `W.LEAD:G.HRS` interaction term.

In [ ]:
crossLevel <- lme(WBEING ~ W.HRS + W.LEAD * G.HRS + G.LEAD, random = ~ W.LEAD | GRP,
                  data = bh1996, control = ctl)
round(summary(crossLevel)$tTable, 4)

In [ ]:
# predicted leadership slope for teams at low (10th percentile) vs high (90th) average hours
teamHrs <- tapply(bh1996$HRS, bh1996$GRP, mean)
teamLeadAvg <- mean(tapply(bh1996$LEAD, bh1996$GRP, mean))
grid <- expand.grid(W.LEAD = seq(-1.5, 1.5, by = 0.1), G.HRS = quantile(teamHrs, c(0.1, 0.9)))
grid$W.HRS <- 0
grid$G.LEAD <- teamLeadAvg
grid$predicted <- predict(crossLevel, newdata = grid, level = 0)
grid$teamHours <- factor(round(grid$G.HRS, 1), labels = c("shorter-hours teams (10th pct)", "longer-hours teams (90th pct)"))

ggplot(grid, aes(W.LEAD, predicted, colour = teamHours)) +
  geom_line(linewidth = 1) +
  labs(x = "leadership rating relative to own team", y = "predicted well-being", colour = NULL,
       title = "Leadership slope in shorter- vs. longer-hours teams")

**Your interpretation.** The interaction estimate is positive with p ≈ .07. Write what you'd tell a customer in 3–4 sentences, then check yourself against these questions:

- Did you claim the effect exists, or describe it as a pattern worth checking?
- How big is the difference in slopes between the two lines, in plain terms?
- What would make you more confident: another survey cycle, a different set of teams, a pre-registered test?
- Long-hours teams also have lower well-being overall (look at `G.HRS`). Does that change how you'd explain the lines?
- `W.LEAD` dropped from about 0.51 to about 0.10 once the interaction went in. Why? (Hint: with an interaction, `W.LEAD`'s coefficient is the slope when `G.HRS` = 0. What would a team averaging zero hours a day be?) How would you fix it?

*Your interpretation:*

## Wrap-up

In your own words, two sentences each:

1. Why one ordinary regression slope for `LEAD` answers neither the within question nor the between question
2. What a contextual effect is, and what this data says about it
3. Why you center a predictor before giving it a random slope

*Your answers:*

1.

2.

3.